In [1]:

!pip install wfdb matplotlib scipy seaborn

import wfdb
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import firwin, lfilter, butter
import seaborn as sns
sns.set_style("darkgrid")

print("works")

works


In [ ]:

def add_noise(ecg, fs, level="moderate"):
    duration = len(ecg) / fs
    time = np.linspace(0, duration, len(ecg))

    baseline = np.sin(2 * np.pi * 0.3 * time)
    powerline = np.sin(2 * np.pi * 50 * time)
    gaussian = np.random.randn(len(ecg))

    if level == "low":
        noise = 0.1*baseline + 0.02*powerline + 0.05*gaussian
    elif level == "moderate":
        noise = 0.3*baseline + 0.05*powerline + 0.1*gaussian
    else:
        noise = 0.6*baseline + 0.1*powerline + 0.25*gaussian

    return ecg + noise


# FIR Filter
def fir_bandpass_filter(signal, fs, low=0.5, high=40, numtaps=101):
    nyquist = fs / 2
    coeffs = firwin(numtaps, [low/nyquist, high/nyquist], pass_zero=False)
    filtered = lfilter(coeffs, 1.0, signal)
    return filtered


# Butterworth (IIR) Filter
def butter_bandpass_filter(signal, fs, low=0.5, high=40, order=4):
    nyquist = fs / 2
    low_norm = low / nyquist
    high_norm = high / nyquist

    b, a = butter(order, [low_norm, high_norm], btype='band')
    filtered = lfilter(b, a, signal)
    return filtered


# Plot Function (3 graphs)
def plot_ecg(time, noisy, fir, butter, label, noise_level):
    plt.figure(figsize=(13,8))

    # Noisy Signal
    plt.subplot(3,1,1)
    plt.plot(time, noisy, color='r')
    plt.title(f"{label}: Noisy ECG ({noise_level.capitalize()} Noise)")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (mV)")

    # FIR Filtered
    plt.subplot(3,1,2)
    plt.plot(time, fir, color='g')
    plt.title("FIR Filtered ECG")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (mV)")

    # Butterworth Filtered
    plt.subplot(3,1,3)
    plt.plot(time, butter, color='b')
    plt.title("Butterworth (IIR) Filtered ECG")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (mV)")

    plt.tight_layout()
    plt.show()


datasets = {
    "Record 100": ("100", "low"),
    "Record 101": ("101", "moderate"),
    "Record 118": ("118", "high")
}


for label, (record_id, noise_level) in datasets.items():
    record = wfdb.rdrecord(record_id, pn_dir='mitdb')
    fs = int(record.fs)
    ecg = record.p_signal[:, 0]

    duration = 10
    n_samples = duration * fs
    ecg = ecg[:n_samples]

    time = np.linspace(0, duration, n_samples)

    # Add noise
    ecg_noisy = add_noise(ecg, fs, level=noise_level)

    # Apply filters
    ecg_fir = fir_bandpass_filter(ecg_noisy, fs)
    ecg_butter = butter_bandpass_filter(ecg_noisy, fs)

    # Plot comparison
    plot_ecg(time, ecg_noisy, ecg_fir, ecg_butter, label, noise_level)